## Read the first few lines of the Amazon Snap File to understand the structure

In [24]:
import pandas as pd
import gzip
from urllib.request import urlopen
import io

url = "https://snap.stanford.edu/data/bigdata/amazon/amazon-meta.txt.gz"

with urlopen(url) as response:
#with gzip.open(file_path, 'rt', encoding='latin-1') as f:
    with gzip.open(response, mode='rt', encoding='latin-1') as f:
        for i in range(22):
            print(f.readline().strip())

# Full information about Amazon Share the Love products
Total items: 548552

Id:   0
ASIN: 0771044445
discontinued product

Id:   1
ASIN: 0827229534
title: Patterns of Preaching: A Sermon Sampler
group: Book
salesrank: 396585
similar: 5  0804215715  156101074X  0687023955  0687074231  082721619X
categories: 2
|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Preaching[12368]
|Books[283155]|Subjects[1000]|Religion & Spirituality[22]|Christianity[12290]|Clergy[12360]|Sermons[12370]
reviews: total: 2  downloaded: 2  avg rating: 5
2000-7-28  cutomer: A2JW67OY8U6HHK  rating: 5  votes:  10  helpful:   9
2003-12-14  cutomer: A2VE83MZF98ITY  rating: 5  votes:   6  helpful:   5

Id:   2
ASIN: 0738700797


## Download the data from SNAP and store in a dataframe and save to a Parquet file with compression

In [34]:
import pandas as pd
import gzip
from urllib.request import urlopen
import io

url = "https://snap.stanford.edu/data/bigdata/amazon/amazon-meta.txt.gz"
#file_path = r"../data/amazon-meta.txt.gz"


products = []
current_item = {}

with urlopen(url) as response:
#with gzip.open(file_path, 'rt', encoding='latin-1') as f:
    with gzip.open(response, mode='rt', encoding='latin-1') as f:
        for line in f:
            line = line.strip()

            if line.startswith("Id:"):
                if current_item:
                    products.append(current_item)
                current_item = {'Id': line.split(":")[1].strip(),'Active': 'Y'}
            elif line.startswith("discontinued"):
                current_item['Active'] = 'N'
            elif line.startswith("ASIN"):
                current_item['ASIN'] = line.split(":")[1].strip()
            elif line.startswith("title"):
                current_item['Title'] = line.split(":", 1)[1].strip()
            elif line.startswith("group"):
                current_item['Group'] = line.split(":")[1].strip()
            elif line.startswith("salesrank"):
                current_item['Salesrank'] = line.split(":")[1].strip()
            elif line.startswith('similar:'):
                parts = line.split()  # Splits by any whitespace
                current_item["similarCount"] = parts[1]
                current_item['SimilarASINs'] = parts[2:]
            elif line.startswith('|'):
                if 'Categories' not in current_item:
                    current_item['Categories'] = []
                    current_item['CleanCategories'] = []
                current_item['Categories'].append(line)
                # 4 levels
                line_clean = [cat.split('[')[0] for cat in line.strip('|').split('|')][:4]
                current_item['CleanCategories'].append(" > ".join(line_clean))
            elif line.startswith('reviews:'):
                parts = line.split()
                try:
                    current_item['reviews_count'] = int(parts[2])
                    current_item['avg_rating'] = float(parts[-1])
                except (ValueError, IndexError):
                    current_item['reviews_count'] = 0
                    current_item['avg_rating'] = 0.0
        if current_item:
            products.append(current_item)

df = pd.DataFrame(products)
df.to_parquet('../data/amazon_data.parquet', compression='gzip')
print(df.head())

  Id Active        ASIN                                              Title  \
0  0      N  0771044445                                                NaN   
1  1      Y  0827229534            Patterns of Preaching: A Sermon Sampler   
2  2      Y  0738700797                         Candlemas: Feast of Flames   
3  3      Y  0486287785   World War II Allied Fighter Planes Trading Cards   
4  4      Y  0842328327  Life Application Bible Commentary: 1 and 2 Tim...   

  Group Salesrank similarCount  \
0   NaN       NaN          NaN   
1  Book    396585            5   
2  Book    168596            5   
3  Book   1270652            0   
4  Book    631289            5   

                                        SimilarASINs  \
0                                                NaN   
1  [0804215715, 156101074X, 0687023955, 068707423...   
2  [0738700827, 1567184960, 1567182836, 073870052...   
3                                                 []   
4  [0842328130, 0830818138, 0842330313, 084232

## Read the Parquet File and load to a Dataframe

In [2]:
# pip install fastparquet
import pandas as pd

file_path = r"../data/amazon_data.parquet"
df = pd.read_parquet(file_path)
print(df.head())
print("Data loaded successfully!")

  Id Active        ASIN                                              Title  \
0  0      N  0771044445                                               None   
1  1      Y  0827229534            Patterns of Preaching: A Sermon Sampler   
2  2      Y  0738700797                         Candlemas: Feast of Flames   
3  3      Y  0486287785   World War II Allied Fighter Planes Trading Cards   
4  4      Y  0842328327  Life Application Bible Commentary: 1 and 2 Tim...   

  Group Salesrank similarCount  \
0  None      None         None   
1  Book    396585            5   
2  Book    168596            5   
3  Book   1270652            0   
4  Book    631289            5   

                                        SimilarASINs  \
0                                               None   
1  [0804215715, 156101074X, 0687023955, 068707423...   
2  [0738700827, 1567184960, 1567182836, 073870052...   
3                                                 []   
4  [0842328130, 0830818138, 0842330313, 084232

In [3]:
df['similarCount'] = pd.to_numeric(df['similarCount'], errors='coerce').fillna(0).astype(int)

df_active = df[(df['Active'] == 'Y') & (df['similarCount'] >= 5)].copy()

print(df_active.head())
print("Total Rows:", df['Id'].count())
print("Filtered Rows:", df_active['Id'].count())

  Id Active        ASIN                                              Title  \
1  1      Y  0827229534            Patterns of Preaching: A Sermon Sampler   
2  2      Y  0738700797                         Candlemas: Feast of Flames   
4  4      Y  0842328327  Life Application Bible Commentary: 1 and 2 Tim...   
5  5      Y  1577943082    Prayers That Avail Much for Business: Executive   
6  6      Y  0486220125  How the Other Half Lives: Studies Among the Te...   

  Group Salesrank  similarCount  \
1  Book    396585             5   
2  Book    168596             5   
4  Book    631289             5   
5  Book    455160             5   
6  Book    188784             5   

                                        SimilarASINs  \
1  [0804215715, 156101074X, 0687023955, 068707423...   
2  [0738700827, 1567184960, 1567182836, 073870052...   
4  [0842328130, 0830818138, 0842330313, 084232861...   
5  [157794349X, 0892749504, 1577941829, 089274956...   
6  [0486401960, 0452283612, 0486229076, 

# Unique Categories

In [4]:
df_active.loc[4, 'CleanCategories']

['Books > Subjects > Religion & Spirituality > Christianity',
 'Books > Subjects > Religion & Spirituality > Christianity',
 'Books > Subjects > Religion & Spirituality > Christianity',
 'Books > Subjects > Religion & Spirituality > Bible & Other Sacred Texts',
 'Books > Subjects > Religion & Spirituality > Christianity']

In [5]:
all_paths = df['CleanCategories'].dropna().explode().unique()
print(f"Total Unique Categories: {len(all_paths)}")

category_to_id = {name: i for i, name in enumerate(all_paths)}

def map_to_ids(cat_list):
    if isinstance(cat_list, list):
        return list(set(category_to_id[cat] for cat in cat_list if cat in category_to_id))
    return []

df_active['CategoryIDs'] = df_active['CleanCategories'].apply(map_to_ids)

print(df_active[['CleanCategories', 'CategoryIDs']].head())

Total Unique Categories: 1289
                                     CleanCategories      CategoryIDs
1  [Books > Subjects > Religion & Spirituality > ...              [0]
2  [Books > Subjects > Religion & Spirituality > ...              [1]
4  [Books > Subjects > Religion & Spirituality > ...           [0, 3]
5  [Books > Subjects > Religion & Spirituality > ...              [0]
6  [Books > Subjects > Arts & Photography > Photo...  [4, 5, 6, 7, 8]


In [6]:
# Range of Salesrank

rank_min = df_active['Salesrank'].min()
rank_max = df_active['Salesrank'].max()

print(f"Salesrank Range: {rank_min} to {rank_max}")

print(df_active['Salesrank'].describe())

Salesrank Range: -1 to 99999
count     337457
unique    252192
top            0
freq          22
Name: Salesrank, dtype: object


# Add New Id field

In [7]:
df_active['New_Id'] = range(len(df_active))

cols = ['New_Id'] + [c for c in df_active.columns if c != 'New_Id']
df_active = df_active[cols]
print(df_active[['New_Id', 'Id', 'ASIN', 'CategoryIDs']].head())

   New_Id Id        ASIN      CategoryIDs
1       0  1  0827229534              [0]
2       1  2  0738700797              [1]
4       2  4  0842328327           [0, 3]
5       3  5  1577943082              [0]
6       4  6  0486220125  [4, 5, 6, 7, 8]


In [8]:
df_final = df_active[['New_Id', 'Title', 'Group', 'Salesrank', 'reviews_count', 'CategoryIDs']].copy()
df_final['Salesrank'] = pd.to_numeric(df_final['Salesrank'], errors='coerce').fillna(0).astype(int)
df_final['reviews_count'] = df_final['reviews_count'].astype(int)
print(df_final.head())


   New_Id                                              Title Group  Salesrank  \
1       0            Patterns of Preaching: A Sermon Sampler  Book     396585   
2       1                         Candlemas: Feast of Flames  Book     168596   
4       2  Life Application Bible Commentary: 1 and 2 Tim...  Book     631289   
5       3    Prayers That Avail Much for Business: Executive  Book     455160   
6       4  How the Other Half Lives: Studies Among the Te...  Book     188784   

   reviews_count      CategoryIDs  
1              2              [0]  
2             12              [1]  
4              1           [0, 3]  
5              0              [0]  
6             17  [4, 5, 6, 7, 8]  


# Encode Title

## SBERT

In [9]:
# pip install -U sentence-transformers
from sentence_transformers import SentenceTransformer, util

/home/hice1/pmaji3/scratch/gnn02/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [10]:
model = SentenceTransformer('all-MiniLM-L6-v2')
embeddings = model.encode(df_final['Title'].tolist(), show_progress_bar=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 713.24it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 10546/10546 [02:04<00:00, 84.88it/s] 


In [ ]:
df_final['Title_Embeddings'] = list(embeddings)
print(df_final[['Title', 'Title_Embeddings']].head())
print(df_final['Title_Embeddings'].size, df_final['Title_Embeddings'].iloc[0].shape)

                                               Title  \
1            Patterns of Preaching: A Sermon Sampler   
2                         Candlemas: Feast of Flames   
4  Life Application Bible Commentary: 1 and 2 Tim...   
5    Prayers That Avail Much for Business: Executive   
6  How the Other Half Lives: Studies Among the Te...   

                                    Title_Embeddings  
1  [-0.037474245, 0.049659807, -0.04645539, -0.03...  
2  [-0.022021107, 0.1393777, 0.023228364, 0.03877...  
4  [-0.08065854, 0.08710222, 0.06329278, -0.02939...  
5  [0.09351609, 0.020177836, 0.014199039, -0.0878...  
6  [0.07082085, -0.043961942, 0.08685487, 0.03206...  
(337457, 7)
337457 (384,)


In [16]:
# Extract the Embedding to a csv .. TODO: Other features to be appended
embeddings_df = pd.DataFrame(df_final['Title_Embeddings'].tolist())
embeddings_df.to_csv('../data/node_features.csv', index=False, header=False)

# Generate the Edge Index

In [ ]:
asin_to_id = pd.Series(df_active.New_Id.values, index=df_active.ASIN).to_dict()
edges = df_active[['New_Id', 'SimilarASINs']].explode('SimilarASINs')
edges['Target_Id'] = edges['SimilarASINs'].map(asin_to_id)
edge_index_df = edges.dropna(subset=['Target_Id']).copy()
edge_index_df['Target_Id'] = edge_index_df['Target_Id'].astype(int)
edge_list = edge_index_df[['New_Id', 'Target_Id']].rename(
    columns={'New_Id': 'source', 'Target_Id': 'target'}
)
print(edge_list.head())
edge_list.to_csv('../data/edge_list.txt', sep=' ', index=False, header=False)

   source  target
1       0   98598
1       0  149337
1       0   72216
1       0  269948
1       0  303812


In [74]:
asin_list = df_active[df_active['New_Id'] == 0]['SimilarASINs'].values[0]
print(asin_list)

print(df_active[df_active['New_Id'] == 98598][['ASIN']])
print(df_active[df_active['New_Id'] == 149337][['ASIN']])
print(df_active[df_active['New_Id'] == 72216][['ASIN']])
print(df_active[df_active['New_Id'] == 269948][['ASIN']])
print(df_active[df_active['New_Id'] == 303812][['ASIN']])

['0804215715', '156101074X', '0687023955', '0687074231', '082721619X']
              ASIN
161555  0804215715
              ASIN
244916  156101074X
              ASIN
118052  0687023955
              ASIN
444232  0687074231
              ASIN
500600  082721619X


# Ratings

In [17]:
ratings_only = df_active['avg_rating'].astype(int)

ratings_only.to_csv('../data/labels.csv', index=False, header=False)


# Generate NPZ

In [29]:
import sys
import os
sys.path.append(os.path.abspath(".."))

from scripts.create_npz import create_npx

create_npx.create_npz_dataset(node_features_file="../data/node_features.csv", labels_file='../data/labels.csv', edges_file='../data/edge_list.txt', num_splits=10, output_file_name='../data/amazon.npz')


Verified Split 0 - Train: 202474, Val: 67491, Test: 67492
no of nodes 337457
no of features per node 384
no of node labels, should matach no of nodes 337457
no of edges 1157952
